# CEFR 3-Band Classification via a 0-100 Score - 2 Methods (baseline)

**Business pipeline (fixed):** each method produces a **0-100 score**, then **2 cut-points**
on that score split it into the **3 bands**.

```
features -> model -> class probabilities -> 0-100 score -> 2 cut-points -> 3 bands
```

**Bands:**

| Band | CEFR levels | Code |
|---|---|---|
| 0 | A1, A2 | `A1-A2` |
| 1 | B1 | `B1` |
| 2 | B2, C1, C2 | `B2-C1-C2` |

**Baseline to beat:** 77% (senior's regression models). **Target:** >=82%.

**This is the simple baseline version:**
- **No hyperparameter tuning** - fixed, sensible defaults.
- **No cross-validation / CV accuracy** shown.
- Reports plain **TRAIN**, **TEST**, and **FULL (train+test)** accuracy.

The two methods are the classic tree ensembles, both on the same Frank-Hall ordinal skeleton:

| # | Method | Family | One-line intuition |
|---|---|---|---|
| 1 | **Ordinal Random Forest** (Frank-Hall) | Bagged trees | Many independent trees, averaged - kills variance |
| 2 | **Ordinal Boosting** (LightGBM + Frank-Hall) | Boosted trees | Trees added in sequence, each fixing the last - kills bias |

## 0. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    cohen_kappa_score, confusion_matrix, classification_report,
)

import lightgbm as lgb

print("lightgbm loaded")

## 1. Load your data  <-- FILL THIS IN

Assign your dataframe to `df`. Nothing else in the notebook needs changing here.

In [ ]:
# TODO: initialise your dataframe here
df = None

# e.g. df = pd.read_csv("your_file.csv")
# e.g. df = pd.read_excel("your_file.xlsx")

## 2. Columns  <-- FILL THIS IN

Put your chosen features (one per feature group, 8-11 total) in `FEATURE_COLS`.
Everything else is metadata: `ciid`, `location`, `split`, and the label.

In [ ]:
# TODO: one feature per feature group (8-11 total)
FEATURE_COLS = []

# OPTIONAL: map each feature to its feature group / assessment section, so the
# importance tables in every method can be reported per section.
# Leave empty to treat every feature as its own section.
FEATURE_GROUPS = {}     # e.g. {"lex_div": "Vocabulary", "mlu": "Grammar"}

# ---- metadata columns (edit names if yours differ) ----
ID_COL       = "ciid"
LOCATION_COL = "location"
SPLIT_COL    = "split"
LABEL_COL    = "cefr"       # TODO: your CEFR label column

# values inside SPLIT_COL that define the sets; anything else is dropped (bad flags)
TRAIN_VALUE = "train"
TEST_VALUE  = "test"

META_COLS = [ID_COL, LOCATION_COL, SPLIT_COL, LABEL_COL]

## 3. Configuration

In [ ]:
RANDOM_STATE = 42

# Band definition: A1,A2 -> 0 | B1 -> 1 | B2,C1,C2 -> 2
BAND_MAP = {
    "A1": 0, "A2": 0,
    "B1": 1,
    "B2": 2, "C1": 2, "C2": 2,
}
BAND_NAMES = ["A1-A2", "B1", "B2-C1-C2"]
N_BANDS = 3

# Score anchors: score = sum_k P(band_k) * anchor_k  -> lands in [0, 100].
# Equally spaced anchors -> the two cut-points, once tuned, give the same accuracy
# regardless of the absolute values; only the printed number changes.
BAND_ANCHORS = np.array([0.0, 50.0, 100.0])

OPTIMIZE_METRIC = "accuracy"     # or "balanced_accuracy" if bands are imbalanced

BASELINE_ACC = 0.77              # senior's regression baseline
TARGET_ACC   = 0.82              # goal

PERM_REPEATS = 20                # permutation-importance repeats

## 4. Build train / test  (shared by both methods)

Rows whose `split` is neither `train` nor `test` (the "bad flags") are dropped.

In [ ]:
assert df is not None, "Assign your dataframe to `df` in section 1."
assert len(FEATURE_COLS) > 0, "Fill FEATURE_COLS in section 2."

missing = [c for c in FEATURE_COLS + META_COLS if c not in df.columns]
assert not missing, f"Columns not found in df: {missing}"


def to_band(s):
    """Map CEFR labels to band 0/1/2. Accepts 6-level strings or already-coded 0/1/2."""
    s = pd.Series(s)
    if s.dtype.kind in "iuf":
        vals = set(pd.unique(s.dropna()))
        if vals <= {0, 1, 2}:
            return s.astype(int).to_numpy()
    key = s.astype(str).str.strip().str.upper().str.replace(" ", "", regex=False)
    mapped = key.map(BAND_MAP)
    bad = key[mapped.isna()].unique()
    assert len(bad) == 0, f"Unmapped label values: {bad}"
    return mapped.astype(int).to_numpy()


split_norm = df[SPLIT_COL].astype(str).str.strip().str.lower()
train_mask = split_norm == TRAIN_VALUE
test_mask  = split_norm == TEST_VALUE
dropped    = (~train_mask & ~test_mask).sum()

train_df = df.loc[train_mask].copy()
test_df  = df.loc[test_mask].copy()

X_train = train_df[FEATURE_COLS].astype(float)
X_test  = test_df[FEATURE_COLS].astype(float)
y_train = to_band(train_df[LABEL_COL])
y_test  = to_band(test_df[LABEL_COL])

print(f"features           : {len(FEATURE_COLS)}  -> {FEATURE_COLS}")
print(f"train / test rows  : {len(X_train)} / {len(X_test)}")
print(f"dropped (bad flags): {dropped}")
print(f"other split values : {sorted(set(split_norm) - {TRAIN_VALUE, TEST_VALUE})}")
print()
print("train band distribution:")
print(pd.Series(y_train).value_counts().sort_index()
        .rename(lambda i: f"{i} = {BAND_NAMES[i]}"))
print()
print("test band distribution:")
print(pd.Series(y_test).value_counts().sort_index()
        .rename(lambda i: f"{i} = {BAND_NAMES[i]}"))

## 5. Core utilities

- `FrankHallOrdinal` - ordinal decomposition: `K-1` binary models for `P(y > k)`, differenced.
- `proba_to_score` - probabilities -> the **0-100 score**.
- `fit_cutpoints` - grid-search the **2 cut-points** (on train scores).
- `evaluate` - runs the full pipeline for one method and reports TRAIN / TEST / FULL accuracy.
- `report_importance` - native + permutation importance, rolled up to sections.

In [ ]:
class FrankHallOrdinal(BaseEstimator, ClassifierMixin):
    """Ordinal decomposition (Frank & Hall, 2001).

    For K ordered classes, fit K-1 binary models P(y > k), then difference the
    cumulative probabilities into per-class probabilities.
    """

    def __init__(self, base_estimator=None):
        self.base_estimator = base_estimator

    def fit(self, X, y):
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        self.K_ = len(self.classes_)
        self.models_ = []
        for k in range(self.K_ - 1):
            yk = (y > self.classes_[k]).astype(int)
            if len(np.unique(yk)) < 2:
                self.models_.append(("const", float(yk[0])))
            else:
                m = clone(self.base_estimator)
                m.fit(X, yk)
                self.models_.append(("model", m))
        return self

    def predict_proba(self, X):
        n = len(X)
        cum = np.zeros((n, self.K_ - 1))
        for k, (kind, m) in enumerate(self.models_):
            cum[:, k] = m if kind == "const" else m.predict_proba(X)[:, 1]
        cum = np.minimum.accumulate(cum, axis=1)   # P(y>k) must be non-increasing

        p = np.zeros((n, self.K_))
        p[:, 0] = 1.0 - cum[:, 0]
        for k in range(1, self.K_ - 1):
            p[:, k] = cum[:, k - 1] - cum[:, k]
        p[:, -1] = cum[:, -1]

        p = np.clip(p, 1e-9, None)
        return p / p.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


def make_pipe(estimator, scale=True):
    steps = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scale", StandardScaler()))
    steps.append(("model", estimator))
    return Pipeline(steps)


def proba_to_score(proba, anchors=BAND_ANCHORS):
    return np.asarray(proba) @ np.asarray(anchors, dtype=float)


def apply_cutpoints(scores, t1, t2):
    scores = np.asarray(scores)
    return np.where(scores <= t1, 0, np.where(scores <= t2, 1, 2))


def _scorer(name):
    return accuracy_score if name == "accuracy" else balanced_accuracy_score


def fit_cutpoints(scores, y, metric=OPTIMIZE_METRIC, n_grid=120):
    """Exhaustive grid search for the 2 thresholds maximising `metric`."""
    scores = np.asarray(scores)
    sc = _scorer(metric)
    cand = np.unique(np.percentile(scores, np.linspace(0, 100, n_grid)))
    best_val, best_cuts = -1.0, (33.3, 66.7)
    for i in range(len(cand) - 1):
        for j in range(i + 1, len(cand)):
            v = sc(y, apply_cutpoints(scores, cand[i], cand[j]))
            if v > best_val:
                best_val, best_cuts = v, (float(cand[i]), float(cand[j]))
    return best_cuts, best_val


RESULTS, FITTED, IMPORTANCE = [], {}, {}


def _metrics(y, pred):
    return dict(
        acc=accuracy_score(y, pred),
        bal_acc=balanced_accuracy_score(y, pred),
        macro_f1=f1_score(y, pred, average="macro"),
        qwk=cohen_kappa_score(y, pred, weights="quadratic"),
        mae=float(np.mean(np.abs(np.asarray(y) - np.asarray(pred)))),
    )


def evaluate(name, model, anchors=BAND_ANCHORS, verbose=True):
    """Fit on train -> probabilities -> 0-100 score -> 2 cut-points -> TRAIN / TEST / FULL.

    Baseline simplicity: the cut-points are fitted on the TRAIN scores. The test set
    is never used to pick them, so `test_acc` is still an honest held-out number.
    (The more rigorous option fits cut-points on out-of-fold scores - see the doc.)
    """
    model.fit(X_train, y_train)
    p_tr, p_te = model.predict_proba(X_train), model.predict_proba(X_test)
    s_tr, s_te = proba_to_score(p_tr, anchors), proba_to_score(p_te, anchors)

    (t1, t2), _ = fit_cutpoints(s_tr, y_train)          # cut-points from TRAIN scores
    pred_tr = apply_cutpoints(s_tr, t1, t2)
    pred_te = apply_cutpoints(s_te, t1, t2)

    y_full = np.concatenate([y_train, y_test])
    pred_full = np.concatenate([pred_tr, pred_te])

    m_te = _metrics(y_test, pred_te)
    train_acc = accuracy_score(y_train, pred_tr)
    full_acc = accuracy_score(y_full, pred_full)
    argmax_acc = accuracy_score(y_test, np.argmax(p_te, axis=1))

    row = dict(method=name,
               train_acc=train_acc, test_acc=m_te["acc"], full_acc=full_acc,
               test_bal_acc=m_te["bal_acc"], macro_f1=m_te["macro_f1"],
               qwk=m_te["qwk"], mae=m_te["mae"],
               argmax_acc=argmax_acc, lift_vs_argmax=m_te["acc"] - argmax_acc,
               cut1=t1, cut2=t2)
    RESULTS.append(row)
    FITTED[name] = dict(model=model, score_test=s_te, pred_test=pred_te,
                        score_train=s_tr, pred_train=pred_tr,
                        proba_test=p_te, proba_train=p_tr, cuts=(t1, t2))

    if verbose:
        flag = "PASS >=82%" if m_te["acc"] >= TARGET_ACC else (
               "beats 77%" if m_te["acc"] >= BASELINE_ACC else "below 77%")
        print(f"  TRAIN accuracy : {train_acc:.3f}")
        print(f"  TEST  accuracy : {m_te['acc']:.3f}   <-- the honest number  [{flag}]")
        print(f"  FULL  accuracy : {full_acc:.3f}   (train+test; inflated, do not quote)")
        print(f"  cut-points {t1:.1f} / {t2:.1f} | bal {m_te['bal_acc']:.3f} | "
              f"QWK {m_te['qwk']:.3f} | argmax {argmax_acc:.3f} (lift {m_te['acc']-argmax_acc:+.3f})")
    return row


print("utilities ready")

### Importance helpers

Two views are produced for **each** method:

- **Native importance** - what the tree ensemble itself exposes (impurity reduction for the
  forest, split gain for LightGBM), reported **per cumulative question** - which sections
  matter at the low end (`P(y>0)`) versus the high end (`P(y>1)`).
- **Permutation importance on the final band prediction** - shuffle one feature, re-run the
  *entire* pipeline (model -> 0-100 score -> cut-points) and measure the drop in band accuracy.
  Model-agnostic and measures the **actual deliverable**, so it is the one to quote.

Both are rolled up to assessment sections via `FEATURE_GROUPS`.

In [ ]:
def banded_predict(model, X, t1, t2, anchors=BAND_ANCHORS):
    """Full pipeline: probabilities -> 0-100 score -> cut-points -> band."""
    return apply_cutpoints(proba_to_score(model.predict_proba(X), anchors), t1, t2)


def permutation_importance_banded(model, X, y, t1, t2, n_repeats=None, seed=RANDOM_STATE):
    """Permutation importance measured on the FINAL band prediction."""
    n_repeats = n_repeats or PERM_REPEATS
    rng = np.random.default_rng(seed)
    base = accuracy_score(y, banded_predict(model, X, t1, t2))
    recs = []
    for col in X.columns:
        drops = []
        for _ in range(n_repeats):
            Xp = X.copy()
            Xp[col] = rng.permutation(Xp[col].to_numpy())
            drops.append(base - accuracy_score(y, banded_predict(model, Xp, t1, t2)))
        recs.append((col, float(np.mean(drops)), float(np.std(drops))))
    return (pd.DataFrame(recs, columns=["feature", "importance", "std"])
              .sort_values("importance", ascending=False).reset_index(drop=True)), base


def frankhall_native_importance(fh, how="gain"):
    """Native importance per cumulative question of a Frank-Hall model."""
    prof = {}
    for k, (kind, m) in enumerate(fh.models_):
        if kind != "model":
            continue
        est = m.steps[-1][1] if hasattr(m, "steps") else m
        vals = None
        if hasattr(est, "booster_"):
            try:
                vals = est.booster_.feature_importance(importance_type=how)
            except Exception:
                vals = None
        if vals is None:
            vals = getattr(est, "feature_importances_", None)
        if vals is not None:
            v = np.asarray(vals, dtype=float)
            prof[f"P(y>{k})"] = v / v.sum() if v.sum() else v
    if not prof:
        return None
    out = pd.DataFrame(prof, index=FEATURE_COLS)
    out["mean"] = out.mean(axis=1)
    return out.sort_values("mean", ascending=False)


def report_importance(name, native=None, native_note=""):
    """Show native + permutation importance for one method, and roll up to sections."""
    fit = FITTED[name]
    t1, t2 = fit["cuts"]
    perm, base = permutation_importance_banded(fit["model"], X_test, y_test, t1, t2)
    perm["section"] = perm["feature"].map(lambda f: FEATURE_GROUPS.get(f, f))
    IMPORTANCE[name] = dict(native=native, permutation=perm)

    if native is not None:
        print(f"NATIVE importance{(' - ' + native_note) if native_note else ''}")
        nat = native.copy()
        nat["section"] = [FEATURE_GROUPS.get(f, f) for f in nat.index]
        display(nat.round(4))

    print(f"\nPERMUTATION importance on the final band prediction "
          f"(baseline test accuracy {base:.3f})")
    print("value = drop in band accuracy when that feature is shuffled")
    display(perm[["feature", "section", "importance", "std"]].round(4))

    weak = perm[perm["importance"] <= 0]["feature"].tolist()
    if weak:
        print(f"contributing nothing here: {weak}")

    if FEATURE_GROUPS:
        sect = (perm.groupby("section")["importance"].agg(["sum", "mean", "count"])
                    .sort_values("sum", ascending=False))
        sect.columns = ["total_importance", "mean_importance", "n_features"]
        print("\nSECTION-level roll-up")
        display(sect.round(4))
    return perm


print("importance helpers ready")

---

# Method 1 - Ordinal Random Forest (Frank-Hall)

*Current best performer.*

### How it works
The three bands are **ordered**, so instead of one 3-class problem we ask **two cumulative
yes/no questions** (Frank-Hall decomposition):

| Model | Question | Positive label |
|---|---|---|
| RF #1 | is this learner **above band 0**? (`y > 0`) | B1, B2-C1-C2 |
| RF #2 | is this learner **above band 1**? (`y > 1`) | B2-C1-C2 |

Each question gets its own Random Forest (hundreds of decision trees, each on a bootstrap
resample of the rows and each split restricted to a random subset of features). The output
probability is the fraction of trees voting "yes".

Writing `c0 = P(y>0)` and `c1 = P(y>1)`, band probabilities come back by differencing:
`P(band0) = 1 - c0`, `P(band1) = c0 - c1`, `P(band2) = c1`.

### Why it suits this problem
Bagging attacks **variance**, the dominant failure mode at n~220, and it is unbothered by the
feature groups being correlated proxies of the same construct.

*Fixed defaults used below - no tuning at this baseline stage.*

In [ ]:
print("Method 1 - Ordinal Random Forest (Frank-Hall)")

rf = RandomForestClassifier(
    n_estimators=800, max_features="sqrt", min_samples_leaf=3,
    class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1,
)
M1 = "1. Ordinal RF (Frank-Hall)"
evaluate(M1, make_pipe(FrankHallOrdinal(rf), scale=False))

In [ ]:
fh_rf = FITTED[M1]["model"].steps[-1][1]
report_importance(M1,
                  native=frankhall_native_importance(fh_rf),
                  native_note="impurity reduction, per cumulative question")

---

# Method 2 - Ordinal Boosting (LightGBM + Frank-Hall)

### How it works
Same Frank-Hall skeleton, but each cumulative question is answered by a **gradient-boosted
ensemble**. Boosting builds trees **sequentially** - each new tree is fitted to the errors the
current ensemble still makes, so it steadily self-corrects. Trees are kept shallow
(`max_depth` 2-4) and combined at a low learning rate.

### Bagging vs boosting - the contrast to study
- **Random Forest (bagging):** trees in **parallel**, averaged. Reduces **variance**. More
  trees never overfits.
- **Boosting:** trees in **sequence**, each depending on the last. Reduces **bias**. More
  trees *can* overfit - hence the conservative fixed settings below.

They fail in different ways, so agreement between them is a good sign.

*Fixed defaults used below - no tuning at this baseline stage.*

In [ ]:
print("Method 2 - Ordinal Boosting (LightGBM + Frank-Hall)")

gbm = lgb.LGBMClassifier(
    n_estimators=400, learning_rate=0.03, max_depth=3, num_leaves=7,
    min_child_samples=15, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8, reg_lambda=1.0,
    random_state=RANDOM_STATE, verbose=-1,
)
M2 = "2. Ordinal Boosting (LGBM + Frank-Hall)"
evaluate(M2, make_pipe(FrankHallOrdinal(gbm), scale=False))

In [ ]:
fh_gbm = FITTED[M2]["model"].steps[-1][1]
report_importance(M2,
                  native=frankhall_native_importance(fh_gbm, how="gain"),
                  native_note="LightGBM gain, per cumulative question")

---

# Results

## Leaderboard

| Column | Meaning | Honest? |
|---|---|---|
| `train_acc` | in-sample: fitted on train, predicted on train | **No** - optimistic |
| `test_acc`  | held-out test set | **Yes** - the number to report |
| `full_acc`  | train + test combined | **No** - inflated by the in-sample train part |

- **Report `test_acc`** against the 77% baseline. `train_acc` and `full_acc` are diagnostics.
- A big `train_acc - test_acc` gap just means the model fits its training data tightly
  (normal for tree ensembles); what matters is that `test_acc` holds up.

In [ ]:
lb = (pd.DataFrame(RESULTS).sort_values("test_acc", ascending=False)
        .reset_index(drop=True))
lb["vs_baseline"] = lb["test_acc"] - BASELINE_ACC
lb["status"] = np.where(lb["test_acc"] >= TARGET_ACC, "PASS >=82%",
                 np.where(lb["test_acc"] >= BASELINE_ACC, "beats 77%", "below 77%"))
lb["train_test_gap"] = lb["train_acc"] - lb["test_acc"]

pd.set_option("display.width", 200, "display.max_columns", 50)
display(lb[["method", "train_acc", "test_acc", "full_acc", "train_test_gap",
            "test_bal_acc", "macro_f1", "qwk", "mae",
            "argmax_acc", "lift_vs_argmax", "cut1", "cut2",
            "vs_baseline", "status"]].round(4))

print(f"\nbaseline {BASELINE_ACC:.0%} | target {TARGET_ACC:.0%}")
print(f"best (by TEST accuracy): {lb.iloc[0]['method']} -> {lb.iloc[0]['test_acc']:.3f}")

## Does the 0-100 score beat plain `argmax`?

Positive `lift_vs_argmax` means the score + cut-points classify better than reading the
model's raw top prediction - i.e. the business 0-100 step is also earning its keep.

In [ ]:
crux = lb[["method", "test_acc", "argmax_acc", "lift_vs_argmax"]].sort_values(
    "lift_vs_argmax", ascending=False)
display(crux.round(4))

## Feature importance across both methods

In [ ]:
comp = pd.DataFrame(index=FEATURE_COLS)
for name, d in IMPORTANCE.items():
    comp[name] = d["permutation"].set_index("feature")["importance"]
comp["mean"] = comp.mean(axis=1)
comp["agreement"] = (comp[list(IMPORTANCE)] > 0).sum(axis=1)
comp["section"] = [FEATURE_GROUPS.get(f, f) for f in comp.index]
comp = comp.sort_values("mean", ascending=False)

print("Permutation importance (drop in band accuracy) - both methods side by side")
print("'agreement' = how many of the 2 methods found the feature useful\n")
display(comp.round(4))

always = comp[comp["agreement"] == len(IMPORTANCE)].index.tolist()
never = comp[comp["agreement"] == 0].index.tolist()
print(f"useful in BOTH methods : {always}")
print(f"useful in NEITHER      : {never}   <- drop candidates")

if FEATURE_GROUPS:
    print("\nSection-level, averaged across methods:")
    display(comp.groupby("section")["mean"].agg(["sum", "mean", "count"])
                .sort_values("sum", ascending=False).round(4))

## Confusion matrix for the best method

In [ ]:
best_name = lb.iloc[0]["method"]
best = FITTED[best_name]
print(f"=== {best_name} ===  cut-points: {best['cuts'][0]:.1f}, {best['cuts'][1]:.1f}\n")
display(pd.DataFrame(confusion_matrix(y_test, best["pred_test"]),
                     index=[f"true {b}" for b in BAND_NAMES],
                     columns=[f"pred {b}" for b in BAND_NAMES]))
print(classification_report(y_test, best["pred_test"], target_names=BAND_NAMES))

## Soft-voting ensemble of the two

Averaging the probability vectors of the forest (low variance) and the booster (low bias)
often nudges accuracy up - but on a small test set, verify it actually helps.

In [ ]:
members = list(FITTED)
P_tr = np.mean([FITTED[m]["proba_train"] for m in members], axis=0)
P_te = np.mean([FITTED[m]["proba_test"] for m in members], axis=0)

s_tr, s_te = proba_to_score(P_tr), proba_to_score(P_te)
(t1e, t2e), _ = fit_cutpoints(s_tr, y_train)

pred_tr, pred_te = apply_cutpoints(s_tr, t1e, t2e), apply_cutpoints(s_te, t1e, t2e)
m = _metrics(y_test, pred_te)

print(f"ENSEMBLE of {len(members)}: {members}")
print(f"  TRAIN {accuracy_score(y_train, pred_tr):.3f} | TEST {m['acc']:.3f}")
print(f"  bal {m['bal_acc']:.3f} | macroF1 {m['macro_f1']:.3f} | QWK {m['qwk']:.3f}")
print(f"  cut-points ({t1e:.1f}, {t2e:.1f})")
print(f"\nbest single method: {lb.iloc[0]['test_acc']:.3f} "
      f"-> ensemble: {m['acc']:.3f}  ({m['acc'] - lb.iloc[0]['test_acc']:+.3f})")

## Final 0-100 scores for the test set

The business deliverable: a per-row 0-100 score plus the band it falls into.

In [ ]:
out = pd.DataFrame({
    ID_COL:       test_df[ID_COL].values,
    LOCATION_COL: test_df[LOCATION_COL].values,
    "score_0_100": np.round(FITTED[best_name]["score_test"], 2),
    "pred_band":   [BAND_NAMES[i] for i in FITTED[best_name]["pred_test"]],
    "true_band":   [BAND_NAMES[i] for i in y_test],
})
out["correct"] = out["pred_band"] == out["true_band"]
display(out.head(20))

print(f"\nmethod: {best_name}")
print(f"cut-points: {FITTED[best_name]['cuts'][0]:.1f} / {FITTED[best_name]['cuts'][1]:.1f}")
print(f"test accuracy: {out['correct'].mean():.3f}")

# out.to_csv("test_scores.csv", index=False)

---

# Appendix - the 0-100 conversion and the split points

## A.1 From the two scores to a 0-100 number

Each method outputs **two** numbers, one per Frank-Hall question:

- `c0 = P(y > 0)` - probability the learner is **above A1-A2** (B1 or higher)
- `c1 = P(y > 1)` - probability the learner is **above B1** (B2-C1-C2)

**Step 1 - two scores to three band probabilities** (differencing):
```
P(A1-A2)    = 1  - c0
P(B1)       = c0 - c1
P(B2-C1-C2) =      c1        (sums to 1)
```

**Step 2 - three probabilities to one score** (weighted average over anchors 0/50/100):
```
score = P(A1-A2)*0 + P(B1)*50 + P(B2-C1-C2)*100
```

**Step 3 - the algebra collapses:**
```
score = (1-c0)*0 + (c0-c1)*50 + c1*100 = 50*(c0 + c1)
      = 50 * ( P(y>0) + P(y>1) )
```

**Why it is always in [0, 100].** `c0, c1` are probabilities in [0,1], so `c0+c1` is in [0,2]
and `*50` lands in [0,100]. (Equivalently, a weighted average can never leave the range of
the anchors it averages.)

**Intuition.** Each of the two proficiency bars a learner clears is worth 50 points, weighted
by the model's confidence they cleared it. Worked example, `c0=0.90, c1=0.30`:
`50*(0.90+0.30) = 60`, matching `P=(0.10,0.60,0.30) -> 0*.1+50*.6+100*.3 = 60`.

**Why an expected value, not `argmax`.** `argmax` discards confidence: two learners both
labelled "B1" (scores 30 vs 70) look identical to it. The 0-100 score separates them, which
is exactly what the business wants - and it lets the cut-points recover accuracy `argmax`
leaves behind (the `lift_vs_argmax` column).

## A.2 How the two split points are decided

**Why not fixed 33.3 / 66.7?** That assumes scores are spread evenly and the model is
perfectly calibrated. In reality scores clump and are shifted, so a fixed 66.7 can slice
through a dense cluster and misclassify a whole group. The cut-points must sit where the model
**actually** separates the bands - an empirical question.

**The search.**
1. Take 120 **percentiles** of the score distribution as candidate positions (so candidates
   land where the learners actually are).
2. Try **every** ordered pair `(t1 < t2)` - ~7,000 combinations.
3. Score each by accuracy of `band = 0 if s<=t1, 1 if s<=t2, else 2`.
4. Keep the best pair. Exhaustive, so it is the global optimum; `33/66` is just one of the
   pairs it tries, so the result is always at least as good.

**Baseline choice: cut-points fit on TRAIN scores.** For simplicity here they are chosen on
the training scores. The **test set is never used to pick them**, so `test_acc` remains an
honest held-out number - at worst the cut-points are slightly sub-optimal (a conservative
baseline, never an inflated one). The more rigorous upgrade fits them on out-of-fold
predictions; that is explained in full in `docs/CEFR_2_Methods_Explained.md` (sections 6, and
the OOF walk-through). Switch to it once you move past baseline.

## A.3 Is the optimum a plateau or a spike?
A single best pair means little if accuracy collapses when a threshold moves by a point.
The next cell measures how wide the near-optimal region is: a **broad plateau** means stable
thresholds that transfer; a **narrow spike** means they are fitted to noise.

In [ ]:
sel = best_name
s_best = FITTED[sel]["score_train"]
t1b, t2b = FITTED[sel]["cuts"]
cand = np.unique(np.percentile(s_best, np.linspace(0, 100, 120)))

rows = []
for a in range(len(cand) - 1):
    for b in range(a + 1, len(cand)):
        rows.append((cand[a], cand[b],
                     accuracy_score(y_train, apply_cutpoints(s_best, cand[a], cand[b]))))
grid = (pd.DataFrame(rows, columns=["t1", "t2", "train_acc"])
          .sort_values("train_acc", ascending=False).reset_index(drop=True))

print(f"{sel}: searched {len(grid):,} threshold pairs\n")
display(grid.head(10).round(3))

best_v = grid["train_acc"].iloc[0]
w1 = (grid["train_acc"] >= best_v - 0.01).sum()
print(f"best train accuracy  : {best_v:.3f}")
print(f"pairs within 1 point : {w1:,}  ({w1/len(grid):.1%} of the grid)")
print(f"chosen cut-points    : {t1b:.2f} / {t2b:.2f}")

top = grid[grid["train_acc"] >= best_v - 0.01]
print(f"\nplateau spans t1 in [{top['t1'].min():.1f}, {top['t1'].max():.1f}], "
      f"t2 in [{top['t2'].min():.1f}, {top['t2'].max():.1f}]")
print("wide plateau => stable thresholds; narrow spike => fitted to noise.")